# Aegis · Colab node
Run one copy per node (node-1 through node-5). Synthetic demo only. Use for the live session, then stop and disconnect. Keep keys out of saved cells/output. See SETUP.md. Respect Colab runtime quotas; local mode is the fallback.

In [ ]:
from google.colab import files
import io, zipfile
from pathlib import Path
uploaded = files.upload()  # cloud-security-demo.zip only
archive = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    root = Path('/content').resolve()
    for member in z.infolist():
        if not (root / member.filename).resolve().is_relative_to(root):
            raise ValueError('Unsafe ZIP path')
    z.extractall(root)
%cd /content/cloud-security-demo
!pip -q install -r node-agent/requirements.txt


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb
!cloudflared --version


## Configure and start
Change only NODE_NAME below in each copy. Enter the matching node-derived secret at the hidden prompt; never enter MASTER_SECRET. Each notebook automatically discovers/registers its own tunnel URL. Stop any previous process before re-running this cell.

In [ ]:
import getpass, os, subprocess, sys
NODE_NAME = 'node-1'  # node-1, node-2, node-3, node-4, node-5
MASTER_URL = input('Your Render HTTPS URL: ').strip().rstrip('/')
NODE_SECRET = getpass.getpass('Matching per-node credential: ')
if 'agent_process' in globals() and agent_process.poll() is None:
    raise RuntimeError('Stop the previous agent with the cleanup cell before restarting.')
env = {**os.environ, 'NODE_NAME': NODE_NAME, 'MASTER_URL': MASTER_URL,
       'NODE_SECRET': NODE_SECRET, 'AGENT_PORT': '6000', 'PYTHONUNBUFFERED': '1'}
env.pop('LOCAL_DEMO', None)
env.pop('PUBLIC_URL', None)
log = open('/content/agent.log', 'w')
agent_process = subprocess.Popen([sys.executable, 'node-agent/agent.py'], env=env,
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('Agent process started:', agent_process.pid)


In [ ]:
print('Running:', agent_process.poll() is None)
print(open('/content/agent.log').read()[-4000:])


## Stop after demo
This terminates the process group including cloudflared. Then disconnect/delete the runtime. Do not run as persistent hosting.

In [ ]:
import os, signal
if agent_process.poll() is None:
    os.killpg(agent_process.pid, signal.SIGTERM)
log.close()
print('Stopped. Disconnect the Colab runtime.')
